# 04 Challenge

Merged notebook for the challenge module.

In [ ]:
# @title ⚙️ Environment Setup (Run this first!)
!pip install pymatgen numpy matplotlib scikit-learn -q
print("✅ Packages installed")

In [ ]:
# @title 📂 Load Tutorial Data
import os

REPO = "MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.basename(os.getcwd()) == REPO:
    print("✅ Data already present")
elif os.path.exists(REPO):
    os.chdir(REPO)
    print("✅ Data already present")
else:
    !git clone {REPO_URL} -q
    os.chdir(REPO)
    print("✅ Data loaded successfully")


## 04a - Open Challenge

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt

from tutorial_utils.runners import get_challenge_predictions, run_challenge_baseline
from tutorial_utils.sections import challenge_baseline as s04a


def create_challenge_baseline(
    top_k_to_print=5,
    max_experiment_patterns=None,
    output_dir="outputs/conventional/profile_correlation",
    show_steps=True,
):
    """Run the challenge baseline using profile-correlation matching."""
    return run_challenge_baseline(
        top_k_to_print=top_k_to_print,
        max_experiment_patterns=max_experiment_patterns,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_challenge_match(
    mystery_pattern="data/challenge/mystery_patterns/01.xy",
    reference_dir="data/reference_structures",
    top_k=5,
    show_plot=True,
):
    """Expose challenge baseline internals for one mystery pattern."""
    two_theta, exp_profile = s04a.load_experimental_profile(Path(mystery_pattern))
    ref_lib = s04a.load_reference_stick_library(sorted(Path(reference_dir).glob("*.cif")))
    by_pearson, by_cosine, simulated = s04a.rank_phases(exp_profile, two_theta, ref_lib)

    print(f"Loaded mystery profile with {len(two_theta)} points")
    print(f"Compared against {len(ref_lib)} references")
    print("Top by Pearson:")
    for i, row in enumerate(by_pearson[:top_k], start=1):
        print(f"  {i}. {row['phase']:<16s} score={row['pearson']:.3f}")

    best = by_pearson[0]["phase"]
    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(two_theta, exp_profile, color="black", linewidth=1.8, label="Mystery pattern")
        plt.plot(two_theta, simulated[best], color="#1f4ed8", linewidth=1.8, label=f"Best match: {best}")
        plt.xlabel("2θ")
        plt.ylabel("Normalized intensity")
        plt.title("Challenge inner step: best profile-correlation match")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {
        "two_theta": two_theta,
        "experimental_profile": exp_profile,
        "top_by_pearson": by_pearson[:top_k],
        "top_by_cosine": by_cosine[:top_k],
    }


# 04a — Open Challenge

Use the mystery patterns to test what you learned from the conventional and ML modules.

## Build a Baseline Solver
This cell applies full-profile correlation to each mystery pattern and records the top predicted phase.

In [ ]:
create_challenge_baseline()
predictions = get_challenge_predictions()
predictions[:5]

# Try on your own:
# inspect_challenge_match(mystery_pattern="data/challenge/mystery_patterns/05.xy")


## Compare Against Ground Truth
Ground truth labels can contain phases outside the reference library, so this baseline is intentionally imperfect.

In [ ]:
import csv

truth_map = {}
with open("data/challenge/ground_truth.csv", newline="") as fh:
    for row in csv.DictReader(fh):
        truth_map[row["new_filename"]] = row["old_filename"].replace(".xy", "")
n_exact = 0
for fname, pred in predictions:
    true_label = truth_map.get(fname, "")
    exact = pred == true_label
    n_exact += int(exact)
    print(f"{fname}: predicted={pred:16s} | true={true_label}")
print(f"\nExact top-1 matches: {n_exact}/{len(predictions)}")


## 🧪 Try Your Own Strategy
Try combining multiple methods from earlier modules, adding your own preprocessing, or training a custom model for this challenge set.

## Summary
- The challenge set is designed to expose limitations of simple single-method pipelines.
- Missing reference phases make this a realistic open-set problem.
- Better performance usually comes from combining preprocessing, robust simulation, and model-based ranking.